In [ ]:
import csv
import json

accepted = []
rejected = []


def parse_row(row):
    """Convert selected fields to numeric types."""
    row["age"] = int(row["age"])
    row["fee_inr"] = int(row["fee_inr"])
    row["prior_no_shows"] = int(row["prior_no_shows"])
    row["distance_km"] = float(row["distance_km"])
    return row


with open("appointment.csv", mode="r", newline="",encoding="utf-8") as f:
  reader = csv.DictReader(f)
  print("Header",reader.fieldnames)

  rows = list(reader)
  print("Row count:",len(rows))


  for row in rows:

    try:
      accepted.append(parse_row(row))
    except ValueError:
      rejected.append(row)


# Save rejected rows to JSON
with open("rejected_rows.json", "w") as f:
    json.dump(rejected, f, indent=2)

print("Accepted rows:", len(accepted))
print("Rejected rows:", len(rejected))

In [ ]:
import pandas as pd

dfs = pd.read_csv("appointment.csv")

print("Shape:",dfs.shape)
print("Info:",dfs.info())
print("Is null values per column:",dfs.isnull().sum())
print("Exact duplicate rows:",dfs.duplicated().sum())

# (a) drop exact duplicate rows;
dfs=dfs.drop_duplicates()
print("After dropping Duplicates",len(dfs))

#(b) clean booking_channel with .str.strip().str.title();
dfs["booking_channel"]=dfs["booking_channel"].str.strip().str.title()
print("After cleaning booking_channel",dfs["booking_channel"].unique())

# (c) convert appt_date with pd.to_datetime(..., errors="coerce") and booked_on with pd.to_datetime;
dfs["appt_date"]=pd.to_datetime(dfs["appt_date"],errors="coerce")
dfs["booked_on"]=pd.to_datetime(dfs["booked_on"],errors="coerce")
print("Row count after date conversion",len(dfs))

# (d) drop rows where appt_date is missing after conversion or status is missing
dfs=dfs.dropna(subset=["appt_date","status"])
print("Row count after dropping missing values",len(dfs))

In [ ]:
import numpy as np
import pandas as pd

# (a) Treat any age below 1 or above 100 as invalid and set it to missing.
dfs.loc[(dfs["age"] < 1 ) | (dfs["age"] > 100),"age"] =np.nan

# (b)
dfs["age"]=dfs["age"].fillna(dfs["age"].median()).round().astype(int)
dfs["distance_km"]=dfs["distance_km"].fillna(dfs["distance_km"].median())

# (c)
for col in ["distance_km","fee_inr"]:
  q1=dfs[col].quantile(0.25)
  q3=dfs[col].quantile(0.25)
  iqr=q3-q1

  lowerfence=q1-1.5*iqr
  upperfence=q3+1.5*iqr

  capcount=((dfs[col]<lowerfence) | (dfs[col]> upperfence)).sum()
  print(f"values capped in {col}:{capcount}")

  dfs[col]=np.clip(dfs[col],lowerfence,upperfence)

# (d)

dfs["distance_km"]=dfs["distance_km"].round(1)
dfs["fee_inr"]=dfs["fee_inr"].round().astype(int)


In [ ]:
import numpy as np

#Lead days
dfs["lead_days"]=(dfs["appt_date"] - dfs["booked_on"]).dt.days
dfs["no_show"]=np.where(dfs["status"]=="No-Show",1,0)

# Converr columns to 2d numpy array
cols=["age","distance_km","fee_inr","lead_days"]
x=dfs[cols].to_numpy()


#(a) the mean of each column (axis=0);

colmean=np.mean(x,axis=0)
print("column means:",colmean)

# (b)

leadmask=x[:,3]>14
count_gt_14=leadmask.sum()
share_gt_14=leadmask.mean()

print(f"rows with lead days > 14: {count_gt_14} (share: {share_gt_14:.4f})")

# (c)
distcol=x[:,1]
z_scores=(distcol - np.mean(distcol)/np.std(distcol))
outliers_z=(np.abs(z_scores)>2).sum()

print(f"rows with |z|>2 for distance_km: {outliers_z}")






In [ ]:
import pandas as pd
import requests

dfclinic = pd.read_csv("clinics.csv")

temps = []

for _, row in dfclinic.iterrows():
    url = "https://api.open-meteo.com/v1/forecast"
    params={
        "latitude": row["latitude"],
        "longitude": row["longitude"],
        "hourly": "temperature_2m",
        "forecast_days":1
    }

    tempc=None
    reason=""

    #try up to 2 times

    for atmpts in range(2):
      try:
        response= requests.get(url,params=params,timeout=10)
        if response.status_code==200:
          data=response.json()
          hourly_tmps=data.get("Hourely",{}).get("temperature_2m",[])
          if hourly_tmps:
            tempc=hourly_tmps[0]
            break  #exit retry loop
          else:
            reason="missing hourly temperature data"
        else:
          reason=f"HTTP {response.status_code}"
      except requests.exceptions.RequestException as e:
        reason=str(e)

    if tempc is None:
      print(f"Failed for {row['clinic_name']}:{reason}")

    temps.append(tempc)

dfclinic["temp_c"]=temps
dfclinic.to_csv("clinics_enriched.csv",index=False)


In [ ]:
import pandas as pd
import numpy as np


#df = pd.read_csv("appointment.csv")

#Sorting
dfs=dfs.sort_values("appointment_id").copy()

dfs["booked_on"] = pd.to_datetime(dfs["booked_on"], errors="coerce")
dfs["appt_date"] = pd.to_datetime(dfs["appt_date"], errors="coerce")

# Remove rows with invalid dates
dfs = dfs.dropna(subset=["booked_on", "appt_date"])

# Calculate lead days
dfs["lead_days"] = (dfs["appt_date"] - dfs["booked_on"]).dt.days

dfs["no_show"]=np.where(dfs["status"]=="No-Show",1,0)
dfs["booked_on"]=pd.to_datetime(dfs["booked_on"]).dt.strftime("%Y-%m-%d")
dfs["appt_date"] = pd.to_datetime(dfs["appt_date"], errors="coerce")

# Remove rows where appt_date is invalid
dfs = dfs[dfs["appt_date"].notna()]

dfs["appt_date"]=pd.to_datetime(dfs["appt_date"]).dt.strftime("%Y-%m-%d")

print(dfs.head)

# keeping columns
cols = [
    "appointment_id",
    "patient_id",
    "clinic_id",
    "booked_on",
    "appt_date",
    "age",
    "gender",
    "distance_km",
    "fee_inr",
    "booking_channel",
    "visit_type",
    "reminder_sent",
    "prior_no_shows",
    "status",
    "lead_days",
    "no_show"
]

dfclean=dfs[cols]

# validate uniqueness
isunique=dfclean["appointment_id"].is_unique
hasnonulls=not dfclean.isna().any().any()
print(f"appointment id is unique: {isunique}")
print(f"appointment id has no nulls: {hasnonulls}")

# save clean csv
dfclean.to_csv("app_clean.csv", index=False)

clinicsenrdf=pd.read_csv("clinics.csv")


def format_sql_val(val):
  if pd.isna(val):
    return "NULL"
  elif isinstance(val, (int,float)):
    return str(val)
  else:
    return f"'{val}'"

with open("load_data.sql","w",encoding="utf-8") as sqlfile:
  for _, row in clinicsenrdf.iterrows():
    values=", ".join(format_sql_val(val) for val in row)
    print(values)
    sqlfile.write(f"INSERT INTO clinics VALUES ({values});\n")

    for _, row in dfclean.iterrows():
      values=", ".join(format_sql_val(val) for val in row)
      sqlfile.write(f"INSERT INTO appointments VALUES ({values});\n")


